In [23]:
import pandas as pd
import numpy as np

from pathlib import Path

from sklearn.model_selection import train_test_split

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from sklearn.impute import SimpleImputer

from sklearn.preprocessing import (
    StandardScaler,
    OrdinalEncoder,
    OneHotEncoder
)

from sklearn.base import BaseEstimator, TransformerMixin

In [24]:
DATA_DIR = Path("../data/processed")

DATA_FILE = DATA_DIR / "student_mental_health_cleaned.csv"

print("Chemin du fichier :", DATA_FILE.resolve())

Chemin du fichier : C:\Users\ajbba\Desktop\YOUCODE-IA\Mental_Health_Risk_Prediction\data\processed\student_mental_health_cleaned.csv


In [25]:
df = pd.read_csv(DATA_FILE)

print("Dataset chargé avec succès.")

print("Dimensions du Dataset :", df.shape)

display(df.head())

Dataset chargé avec succès.
Dimensions du Dataset : (2000, 16)


,Age,Gender,Year_of_Study,Sleep_Hours,Study_Hours_Per_Week,GPA,Social_Media_Hours,Screen_Time_Hours,Physical_Activity,Financial_Stress,Family_Support,Attendance_Rate,Part_Time_Job,Diet_Quality,Previous_Mental_Health_History,Mental_Health_Risk
0,22,Male,3,6.6,17.6,2.91,6.9,7.5,Moderate,8.5,4.0,78.0,No,Poor,No,High
1,21,Male,4,5.6,22.4,2.92,7.6,11.5,Low,4.5,5.3,85.4,Yes,Average,No,Medium
2,22,Male,5,7.1,28.1,2.68,6.5,8.9,Low,4.8,6.1,79.5,No,Average,No,Medium
3,24,Male,1,4.5,14.8,2.23,5.5,7.6,High,3.7,8.0,73.9,No,Average,Yes,High
4,20,Female,1,6.7,17.2,2.69,4.3,7.0,Moderate,8.0,7.4,99.8,Yes,NaN,No,Medium


In [26]:
TARGET_COLUMN = "Mental_Health_Risk"

X = df.drop(columns=[TARGET_COLUMN]).copy()

y = df[TARGET_COLUMN].copy()

print("Dimensions de X :", X.shape)

print("Dimensions de y :", y.shape)

Dimensions de X : (2000, 15)
Dimensions de y : (2000,)


In [27]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Dimensions de X_train :", X_train.shape)

print("Dimensions de X_test :", X_test.shape)

print("Dimensions de y_train :", y_train.shape)

print("Dimensions de y_test :", y_test.shape)

Dimensions de X_train : (1600, 15)
Dimensions de X_test : (400, 15)
Dimensions de y_train : (1600,)
Dimensions de y_test : (400,)


In [28]:
total_samples = len(X)

train_percentage = len(X_train) / total_samples * 100

test_percentage = len(X_test) / total_samples * 100

print("Nombre total d'observations :", total_samples)

print("Nombre d'observations Train :", len(X_train))

print("Nombre d'observations Test :", len(X_test))

print("Pourcentage Train :", round(train_percentage, 2), "%")

print("Pourcentage Test :", round(test_percentage, 2), "%")

Nombre total d'observations : 2000
Nombre d'observations Train : 1600
Nombre d'observations Test : 400
Pourcentage Train : 80.0 %
Pourcentage Test : 20.0 %


Traiter les valeurs manquantes en choisissant la méthode d’imputation la plus adaptée à la nature des données

In [29]:
missing_train = X_train.isnull().sum()

missing_train = missing_train[missing_train > 0]

print("Variables contenant des valeurs manquantes dans Train :")

display(missing_train)

Variables contenant des valeurs manquantes dans Train :


Sleep_Hours           54
GPA                   36
Social_Media_Hours    48
Financial_Stress      76
Family_Support        36
Attendance_Rate       39
Diet_Quality          55
dtype: int64

In [30]:
missing_test = X_test.isnull().sum()

missing_test = missing_test[missing_test > 0]

print("Variables contenant des valeurs manquantes dans Test :")

display(missing_test)

Variables contenant des valeurs manquantes dans Test :


Sleep_Hours            8
GPA                    2
Social_Media_Hours    10
Financial_Stress      15
Family_Support        18
Attendance_Rate        8
Diet_Quality          14
dtype: int64

Identifier les types de variables

In [31]:
ordinal_features = [
    "Diet_Quality",
    "Physical_Activity"
]

print("Variables ordinales :")

print(ordinal_features)

Variables ordinales :
['Diet_Quality', 'Physical_Activity']


In [ ]:
for column in ordinal_features:

    print(f"Variable : {column}")

    print("Categories dans Train :")

    print(X_train[column].value_counts(dropna=False))

    print("Categories dans Test :")

    print(X_test[column].value_counts(dropna=False))


Variable : Diet_Quality
Catégories dans Train :
Diet_Quality
Average    687
Poor       476
Good       382
NaN         55
Name: count, dtype: int64

Catégories dans Test :
Diet_Quality
Average    173
Poor       111
Good       102
NaN         14
Name: count, dtype: int64

Variable : Physical_Activity
Catégories dans Train :
Physical_Activity
Moderate    742
Low         539
High        319
Name: count, dtype: int64

Catégories dans Test :
Physical_Activity
Moderate    164
Low         140
High         96
Name: count, dtype: int64


In [ ]:
numeric_features = X_train.select_dtypes(
    include=np.number
).columns.tolist()

numeric_features = [
    column
    for column in numeric_features
    if column not in ordinal_features
]

print("Variables numeriques :")

print(numeric_features)

Variables numériques :
['Age', 'Year_of_Study', 'Sleep_Hours', 'Study_Hours_Per_Week', 'GPA', 'Social_Media_Hours', 'Screen_Time_Hours', 'Financial_Stress', 'Family_Support', 'Attendance_Rate']


In [34]:
categorical_features = X_train.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

nominal_features = [
    column
    for column in categorical_features
    if column not in ordinal_features
]

print("Variables nominales :")

print(nominal_features)

Variables nominales :
['Gender', 'Part_Time_Job', 'Previous_Mental_Health_History']


C:\Users\ajbba\AppData\Local\Temp\ipykernel_17676\1756135461.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X_train.select_dtypes(


In [35]:
all_features = (
    numeric_features
    + ordinal_features
    + nominal_features
)

print("Nombre total de variables identifiées :", len(all_features))

print("Nombre de variables dans X_train :", X_train.shape[1])

print(
    "Toutes les variables sont bien réparties :",
    len(all_features) == len(set(all_features))
    and set(all_features) == set(X_train.columns)
)

Nombre total de variables identifiées : 15
Nombre de variables dans X_train : 15
Toutes les variables sont bien réparties : True


Détecter et traiter les valeurs aberrantes en utilisant une méthode appropriée (IQR, Z-score, etc.).

In [36]:
iqr_limits = {}

for column in numeric_features:

    Q1 = X_train[column].quantile(0.25)

    Q3 = X_train[column].quantile(0.75)

    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR

    upper_bound = Q3 + 1.5 * IQR

    iqr_limits[column] = {
        "Q1": Q1,
        "Q3": Q3,
        "IQR": IQR,
        "Lower Bound": lower_bound,
        "Upper Bound": upper_bound
    }

iqr_limits_df = pd.DataFrame(iqr_limits).T

display(iqr_limits_df)

,Q1,Q3,IQR,Lower Bound,Upper Bound
Age,20.00,23.0,3.00,15.50,27.50
Year_of_Study,1.00,4.0,3.00,-3.50,8.50
Sleep_Hours,6.00,7.5,1.50,3.75,9.75
Study_Hours_Per_Week,16.00,24.0,8.00,4.00,36.00
GPA,2.32,2.8,0.48,1.60,3.52
Social_Media_Hours,5.30,6.8,1.50,3.05,9.05
Screen_Time_Hours,7.40,9.7,2.30,3.95,13.15
Financial_Stress,4.50,7.4,2.90,0.15,11.75
Family_Support,5.20,8.0,2.80,1.00,12.20
Attendance_Rate,80.20,90.1,9.90,65.35,104.95


In [37]:
outliers_count = {}

for column in numeric_features:

    lower_bound = iqr_limits[column]["Lower Bound"]

    upper_bound = iqr_limits[column]["Upper Bound"]

    outliers_mask = (
        (X_train[column] < lower_bound)
        |
        (X_train[column] > upper_bound)
    )

    outliers_count[column] = outliers_mask.sum()

outliers_count_df = pd.DataFrame(
    list(outliers_count.items()),
    columns=["Variable", "Nombre_Outliers"]
)

display(outliers_count_df)

,Variable,Nombre_Outliers
0,Age,2
1,Year_of_Study,0
2,Sleep_Hours,9
3,Study_Hours_Per_Week,7
4,GPA,5
5,Social_Media_Hours,10
6,Screen_Time_Hours,7
7,Financial_Stress,0
8,Family_Support,0
9,Attendance_Rate,10


In [ ]:
for column in numeric_features:

    plt.figure(figsize=(8, 4))

    sns.boxplot(x=X_train[column])

    plt.title(f"Détection des valeurs aberrantes : {column}")

    plt.xlabel(column)

    plt.show()

In [38]:
z_scores = pd.DataFrame(
    zscore(
        X_train[numeric_features],
        nan_policy="omit"
    ),
    columns=numeric_features,
    index=X_train.index
)

display(z_scores.head())

,Age,Year_of_Study,Sleep_Hours,Study_Hours_Per_Week,GPA,Social_Media_Hours,Screen_Time_Hours,Financial_Stress,Family_Support,Attendance_Rate
726,-0.063706,-0.471636,-1.381874,-0.238603,0.073072,0.145424,1.103498,0.700067,0.637946,-1.229211
1384,0.875744,-1.232340,-1.649110,-1.330997,-0.965354,0.321819,0.191328,-0.819140,0.366701,-0.459807
1579,-1.003157,-0.471636,0.399702,-0.516037,-1.023044,-0.295565,-0.842465,-0.295275,0.746443,0.762997
1098,0.875744,-1.232340,1.379569,-0.082547,0.880737,0.762808,1.103498,0.804840,-0.935274,-0.212498
582,0.406019,0.289067,0.399702,0.663055,-0.273070,-0.383762,-0.355975,0.647681,0.692195,-1.435302


In [39]:
zscore_outliers = z_scores.abs() > 3

outliers_per_column = zscore_outliers.sum()

print("Nombre d'Outliers par variable :")

display(outliers_per_column)

Nombre d'Outliers par variable :


Age                     2
Year_of_Study           0
Sleep_Hours             0
Study_Hours_Per_Week    3
GPA                     3
Social_Media_Hours      4
Screen_Time_Hours       3
Financial_Stress        0
Family_Support          0
Attendance_Rate         9
dtype: int64

##Vérifier la distribution des classes dans Train

In [42]:
train_class_counts = y_train.value_counts()

train_class_percentages = (
    y_train.value_counts(normalize=True) * 100
)

print("Nombre d'observations par classe dans Train :")

display(train_class_counts)

print("Pourcentage de chaque classe dans Train :")

display(train_class_percentages.round(2))

Nombre d'observations par classe dans Train :


Mental_Health_Risk
High      544
Medium    528
Low       528
Name: count, dtype: int64

Pourcentage de chaque classe dans Train :


Mental_Health_Risk
High      34.0
Medium    33.0
Low       33.0
Name: proportion, dtype: float64

Construire les Pipelines de transformation

In [ ]:
numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),

        ("scaler", StandardScaler())
    ]
)

print("Pipeline numérique cete cree")

Pipeline numérique créé avec succès.


In [ ]:
ordinal_categories = [
    ["Poor", "Average", "Good"],
    ["Low", "Moderate", "High"]
]

ordinal_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),

        ("encoder", OrdinalEncoder(
            categories=ordinal_categories,
            handle_unknown="use_encoded_value",
            unknown_value=-1
        ))
    ]
)

print("Pipeline ordinal ete cree")

Pipeline ordinal créé avec succès.


In [ ]:
nominal_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),

        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ]
)

print("Pipeline nominal ete cree")

Pipeline nominal créé avec succès.


Regrouper les Pipelines avec ColumnTransformer

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, numeric_features),

        ("ord", ordinal_pipeline, ordinal_features),

        ("nom", nominal_pipeline, nominal_features)
    ],
    remainder="drop"
)

print("ColumnTransformer ete cree")

ColumnTransformer créé avec succès.


 le Pipeline global

In [ ]:
preprocessing_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor)
    ]
)

print("Pipeline de pretraitement ete cree.")

Pipeline de prétraitement créé avec succès.


In [49]:
X_train_processed = preprocessing_pipeline.fit_transform(X_train)

print("Transformation de Train terminée.")

print("Dimensions avant transformation :", X_train.shape)

print("Dimensions après transformation :", X_train_processed.shape)

Transformation de Train terminée.
Dimensions avant transformation : (1600, 15)
Dimensions après transformation : (1600, 19)


In [50]:
X_test_processed = preprocessing_pipeline.transform(X_test)

print("Transformation de Test terminée.")

print("Dimensions avant transformation :", X_test.shape)

print("Dimensions après transformation :", X_test_processed.shape)

Transformation de Test terminée.
Dimensions avant transformation : (400, 15)
Dimensions après transformation : (400, 19)


In [51]:
print("Train avant Preprocessing :", X_train.shape)

print("Train après Preprocessing :", X_train_processed.shape)

print("------------------------------------")

print("Test avant Preprocessing :", X_test.shape)

print("Test après Preprocessing :", X_test_processed.shape)

Train avant Preprocessing : (1600, 15)
Train après Preprocessing : (1600, 19)
------------------------------------
Test avant Preprocessing : (400, 15)
Test après Preprocessing : (400, 19)


In [52]:
feature_names = preprocessing_pipeline.named_steps[
    "preprocessor"
].get_feature_names_out()

print("Noms des variables après transformation :")

print(feature_names)

print("\nNombre total de variables après transformation :")

print(len(feature_names))

Noms des variables après transformation :
['num__Age' 'num__Year_of_Study' 'num__Sleep_Hours'
 'num__Study_Hours_Per_Week' 'num__GPA' 'num__Social_Media_Hours'
 'num__Screen_Time_Hours' 'num__Financial_Stress' 'num__Family_Support'
 'num__Attendance_Rate' 'ord__Diet_Quality' 'ord__Physical_Activity'
 'nom__Gender_Female' 'nom__Gender_Male' 'nom__Gender_Non-binary'
 'nom__Part_Time_Job_No' 'nom__Part_Time_Job_Yes'
 'nom__Previous_Mental_Health_History_No'
 'nom__Previous_Mental_Health_History_Yes']

Nombre total de variables après transformation :
19


In [53]:
print(
    "Nombre de Features Train :",
    X_train_processed.shape[1]
)

print(
    "Nombre de Features Test :",
    X_test_processed.shape[1]
)

print(
    "Même nombre de Features :",
    X_train_processed.shape[1] == X_test_processed.shape[1]
)

print(
    "Même nombre de noms de Features :",
    len(feature_names) == X_train_processed.shape[1]
)

Nombre de Features Train : 19
Nombre de Features Test : 19
Même nombre de Features : True
Même nombre de noms de Features : True
